# Manifold Textural MEB — block_0

Explorer si les textures forment des **îlots séparés** ou un **continuum** dans l'espace de features `block_0`.

- UMAP cosine sur PCA-50d
- Densité + ponts entre catégories
- Matrice de connectivité (% voisins cross-catégorie, K=10)
- Validation croisée avec Pairwise Fisher

In [ ]:
# ── Dépendance optionnelle ─────────────────────────────────────────────────────
import subprocess, sys
try:
    import umap
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn'])
    import umap

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import h5py
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

ROOT     = Path('..').resolve()
DB_PATH  = ROOT / 'data' / 'feature_database' / 'database_meb.h5'
CFG_PATH = ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
OUT_DIR  = ROOT / 'outputs'

_mfd_SEED        = 42
_mfd_PCA_DIM     = 50
_mfd_KEY         = 'block_0'
_mfd_CATS_EXCL   = {2, 8, 10, 11, 12, 13}
_mfd_MIN_PATCHES = 30
_mfd_K_NN        = 10

with open(CFG_PATH) as _f:
    _mfd_cfg = json.load(_f)
CATEGORIES = {int(k): v['name']  for k, v in _mfd_cfg['available_categories'].items()}
CAT_COLORS = {int(k): v['color'] for k, v in _mfd_cfg['available_categories'].items()}

with h5py.File(DB_PATH, 'r') as _h5:
    _mfd_IMAGE_NAMES  = _h5['metadata/image_names'][:]
    _mfd_CATEGORY_IDS = _h5['metadata/category_ids'][:].astype(int)
    _mfd_X_all        = _h5['features'][_mfd_KEY][:]

_mfd_CATS_VALID = sorted(
    int(c) for c in np.unique(_mfd_CATEGORY_IDS)
    if int(c) not in _mfd_CATS_EXCL
    and (_mfd_CATEGORY_IDS == int(c)).sum() >= _mfd_MIN_PATCHES
)

_mfd_mask  = np.isin(_mfd_CATEGORY_IDS, _mfd_CATS_VALID)
_mfd_X     = _mfd_X_all[_mfd_mask]
_mfd_y     = _mfd_CATEGORY_IDS[_mfd_mask]
_mfd_imgs  = _mfd_IMAGE_NAMES[_mfd_mask]

# PCA-50d + L2-normalisation
_mfd_pca   = PCA(n_components=_mfd_PCA_DIM, random_state=_mfd_SEED)
_mfd_X_50  = _mfd_pca.fit_transform(_mfd_X)
_mfd_norms = np.linalg.norm(_mfd_X_50, axis=1, keepdims=True)
_mfd_X_norm = _mfd_X_50 / np.where(_mfd_norms < 1e-8, 1.0, _mfd_norms)

print(f'Patches    : {len(_mfd_X)}')
print(f'Catégories : {len(_mfd_CATS_VALID)}')
print(f'PCA variance expliquée : {_mfd_pca.explained_variance_ratio_.sum()*100:.1f}%')
print()
for _c in _mfd_CATS_VALID:
    print(f'  {_c:2d}  {CATEGORIES[_c]:<25}  n={(_mfd_y == _c).sum()}')

## Cell 1 — UMAP du manifold

In [ ]:
_mfd_reducer = umap.UMAP(
    n_neighbors  = 15,
    min_dist     = 0.1,
    n_components = 2,
    metric       = 'cosine',
    random_state = _mfd_SEED,
)
_mfd_X_umap = _mfd_reducer.fit_transform(_mfd_X_norm)   # (N, 2)
print(f'UMAP terminé — shape={_mfd_X_umap.shape}')

# ── Figure principale ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 11))

for _c in _mfd_CATS_VALID:
    _mfd_mask_c = _mfd_y == _c
    ax.scatter(
        _mfd_X_umap[_mfd_mask_c, 0],
        _mfd_X_umap[_mfd_mask_c, 1],
        c=CAT_COLORS[_c],
        label=f'{CATEGORIES[_c]} (n={_mfd_mask_c.sum()})',
        s=12, alpha=0.6, edgecolors='none',
    )

ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
ax.set_title(
    'Manifold textural — block_0\n'
    'UMAP cosine · structure de voisinage réelle\n'
    'amas séparés = îlots · amas connectés = continuum',
    fontsize=12,
)
ax.legend(fontsize=9, markerscale=2, loc='best')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(OUT_DIR / 'manifold_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 2 — UMAP avec densité (ponts entre catégories)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# ── Densité globale (hexbin) ───────────────────────────────────────────────────
hb = ax1.hexbin(
    _mfd_X_umap[:, 0], _mfd_X_umap[:, 1],
    gridsize=40, cmap='viridis', mincnt=1,
)
ax1.set_title(
    'Densité des patches\nzones vides entre amas = îlots séparés',
    fontsize=11,
)
ax1.set_xlabel('UMAP 1')
ax1.set_ylabel('UMAP 2')
plt.colorbar(hb, ax=ax1, label='Nb patches')

# ── Catégories en transparence pour voir les ponts ────────────────────────────
for _c in _mfd_CATS_VALID:
    _mfd_mask_c = _mfd_y == _c
    ax2.scatter(
        _mfd_X_umap[_mfd_mask_c, 0],
        _mfd_X_umap[_mfd_mask_c, 1],
        c=CAT_COLORS[_c],
        label=CATEGORIES[_c],
        s=8, alpha=0.4, edgecolors='none',
    )
ax2.set_title(
    'Ponts entre catégories\npatches mélangés aux frontières = continuum',
    fontsize=11,
)
ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.legend(fontsize=8, markerscale=2, loc='best')

plt.tight_layout()
plt.savefig(OUT_DIR / 'manifold_density.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 3 — Matrice de connectivité entre catégories

Pour chaque paire, % de voisins (K=10 dans l'espace original) issus de l'autre catégorie.

In [ ]:
# ── KNN dans l'espace PCA-50d L2-normalisé ────────────────────────────────────
_mfd_nbrs = NearestNeighbors(
    n_neighbors=_mfd_K_NN + 1, metric='cosine'
).fit(_mfd_X_norm)
_, _mfd_indices = _mfd_nbrs.kneighbors(_mfd_X_norm)   # (N, K+1)

_mfd_n_cats    = len(_mfd_CATS_VALID)
_mfd_cat2idx   = {c: i for i, c in enumerate(_mfd_CATS_VALID)}
_mfd_conn      = np.zeros((_mfd_n_cats, _mfd_n_cats))

for _i in range(len(_mfd_X_norm)):
    _ci = int(_mfd_y[_i])
    for _j in _mfd_indices[_i, 1:]:   # exclure soi-même
        _cj = int(_mfd_y[_j])
        if _ci != _cj:
            _mfd_conn[_mfd_cat2idx[_ci], _mfd_cat2idx[_cj]] += 1

# Normaliser : % de voisins cross-catégorie
for _c in _mfd_CATS_VALID:
    _n_c = (_mfd_y == _c).sum()
    _mfd_conn[_mfd_cat2idx[_c], :] /= (_n_c * _mfd_K_NN)
_mfd_conn *= 100

# ── Heatmap ───────────────────────────────────────────────────────────────────
_mfd_labels = [CATEGORIES[c] for c in _mfd_CATS_VALID]

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(_mfd_conn, cmap='YlOrRd')
ax.set_xticks(range(_mfd_n_cats))
ax.set_yticks(range(_mfd_n_cats))
ax.set_xticklabels(_mfd_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(_mfd_labels, fontsize=9)

_mfd_vmax = _mfd_conn.max()
for _i in range(_mfd_n_cats):
    for _j in range(_mfd_n_cats):
        if _i != _j:
            _mfd_val = _mfd_conn[_i, _j]
            ax.text(
                _j, _i, f'{_mfd_val:.1f}',
                ha='center', va='center', fontsize=8,
                color='white' if _mfd_val > _mfd_vmax * 0.6 else 'black',
            )

plt.colorbar(im, ax=ax, fraction=0.046,
             label="% voisins de l'autre catégorie")
ax.set_title(
    'Connectivité entre textures — block_0\n'
    'élevé = pont/continuum · faible = îlots séparés',
    fontsize=12,
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'manifold_connectivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 4 — Interprétation automatique

In [ ]:
print('=== Connexions entre textures (voisins cross-catégorie) ===\n')

_mfd_paires = []
for _i in range(_mfd_n_cats):
    for _j in range(_i + 1, _mfd_n_cats):
        _conn_sym = (_mfd_conn[_i, _j] + _mfd_conn[_j, _i]) / 2
        _mfd_paires.append((
            _conn_sym,
            CATEGORIES[_mfd_CATS_VALID[_i]],
            CATEGORIES[_mfd_CATS_VALID[_j]],
        ))

_mfd_paires.sort(reverse=True)

print('--- Textures les plus CONNECTÉES (continuum) ---')
for _conn, _c1, _c2 in _mfd_paires[:5]:
    print(f'  {_c1:<25} ↔ {_c2:<25} : {_conn:.1f}%')

print('\n--- Textures les plus ISOLÉES (îlots) ---')
for _conn, _c1, _c2 in _mfd_paires[-5:]:
    print(f'  {_c1:<25} ↔ {_c2:<25} : {_conn:.1f}%')

_mfd_mean_conn = np.mean([_p[0] for _p in _mfd_paires])
print(f'\nConnectivité moyenne inter-catégories : {_mfd_mean_conn:.1f}%')
if _mfd_mean_conn < 5:
    print('→ structure en ÎLOTS (textures bien séparées)')
elif _mfd_mean_conn < 15:
    print('→ structure MIXTE (quelques ponts)')
else:
    print('→ structure en CONTINUUM (textures connectées)')

## Cell 5 — Comparaison connectivité vs Pairwise Fisher

Vérification de cohérence : une paire très connectée dans le manifold devrait avoir un Fisher J faible (classes confondues).

In [ ]:
print('=== Cohérence connectivité ↔ Fisher ===\n')
print('Si une paire est très connectée dans le manifold,')
print('elle devrait avoir un Fisher J faible (confondue).\n')
print('Paire la plus connectée : voir si Fisher faible aussi')
print('→ confirme transition texturale réelle\n')

# Paire Faisceaux / Filaments — connue pour être confondue (Fisher J≈21.7)
_mfd_i_fais = _mfd_cat2idx.get(3)   # Faisceaux
_mfd_i_fila = _mfd_cat2idx.get(4)   # Filaments
if _mfd_i_fais is not None and _mfd_i_fila is not None:
    _mfd_conn_ff = (
        _mfd_conn[_mfd_i_fais, _mfd_i_fila]
        + _mfd_conn[_mfd_i_fila, _mfd_i_fais]
    ) / 2
    print(f'Faisceaux ↔ Filaments : connectivité = {_mfd_conn_ff:.1f}%')
    print(f'  (rappel : Pairwise Fisher J = 21.7 → très confondus)')
    if _mfd_conn_ff > _mfd_mean_conn:
        print('  → cohérent : connectés ET confondus ✅')
    else:
        print('  → incohérent : peu connectés malgré Fisher faible ⚠️')

# Résumé tableau connectivité vs rang Fisher
print('\n--- Top 5 paires les plus connectées ---')
print(f'{"Paire":<52}  {"Conn %":>8}')
print('─' * 62)
for _conn, _c1, _c2 in _mfd_paires[:5]:
    print(f'  {_c1:<24} ↔ {_c2:<24}  {_conn:>6.1f}%')